# Buổi 3 — Embedding và tìm kiếm ngữ nghĩa

Nối tiếp buổi 2 về TF-IDF, cosine similarity và text retrieval. Chạy baseline trước, sau đó so sánh với embedding.

## Chuẩn bị

Cài thư viện nếu môi trường chưa có:

~~~bash
python -m pip install -U scikit-learn sentence-transformers
~~~

Mô hình multilingual sẽ tải ở lần chạy đầu; có thể chạy CPU. Dữ liệu minh họa bằng tiếng Việt.

## 1. Corpus và truy vấn

Đọc nhanh các tài liệu trước khi chạy để biết đoạn nào liên quan đến truy vấn.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ids = ["d1", "d2", "d3", "d4", "d5", "d6"]
docs = [
    "TF-IDF gán trọng số cao cho từ hiếm nhưng quan trọng với tài liệu hiện tại.",
    "Naive Bayes là mô hình xác suất dùng để phân loại email spam.",
    "Vector database lưu embedding, văn bản và metadata để tìm các đoạn gần truy vấn.",
    "Embedding biểu diễn ý nghĩa câu bằng vector số; câu cùng nghĩa thường ở gần nhau.",
    "Cosine similarity đo góc giữa hai vector để xếp hạng tài liệu theo độ giống.",
    "Tiền xử lý văn bản có thể gồm chuẩn hóa chữ thường và tách từ."
]
query = "Làm sao biết chữ nào nói lên nội dung của tài liệu?"

## 2. Baseline TF-IDF

TF-IDF hiệu quả khi truy vấn và tài liệu có từ chung. Ở ví dụ này, truy vấn cố ý diễn đạt khác tài liệu.

In [ ]:
tfidf = TfidfVectorizer()
doc_tfidf = tfidf.fit_transform(docs)

def tfidf_search(query, k=3):
    qv = tfidf.transform([query])
    scores = cosine_similarity(qv, doc_tfidf)[0]
    order = np.argsort(scores)[::-1][:k]
    return [(ids[i], float(scores[i]), docs[i]) for i in order]

for doc_id, score, text in tfidf_search(query):
    print(f"{doc_id} | score={score:.3f} | {text}")

## 3. Tạo embedding và tìm kiếm ngữ nghĩa

Mô hình multilingual hỗ trợ tiếng Việt. Sau khi chuẩn hóa L2, tích vô hướng tương đương cosine similarity.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
doc_embeddings = model.encode(
    docs, convert_to_numpy=True, normalize_embeddings=True
)

def semantic_search(query, k=3):
    query_embedding = model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )[0]
    scores = doc_embeddings @ query_embedding
    order = np.argsort(scores)[::-1][:k]
    return [(ids[i], float(scores[i]), docs[i]) for i in order]

for doc_id, score, text in semantic_search(query):
    print(f"{doc_id} | cosine={score:.3f} | {text}")

## 4. So sánh và bài tập

Thử các truy vấn bên dưới. Đọc nội dung top-3, không chỉ nhìn điểm số.

In [ ]:
comparison_queries = [
    "cách tìm từ đặc trưng trong văn bản",
    "mô hình nào phát hiện thư rác",
    "lưu vector để tìm các đoạn tương tự"
]
for q in comparison_queries:
    print("\nQUERY:", q)
    print("TF-IDF :", [r[0] for r in tfidf_search(q)])
    print("Dense  :", [r[0] for r in semantic_search(q)])

### Bài tập

1. Thêm ít nhất năm tài liệu về chủ đề bạn quan tâm.
2. Tạo truy vấn khớp từ khóa, truy vấn dùng từ đồng nghĩa và truy vấn không có đáp án.
3. So sánh top-3 của TF-IDF và embedding; ghi lại một lỗi.
4. Đề xuất cách phát hiện truy vấn không có kết quả đủ phù hợp.

Embedding có thể giúp với cách diễn đạt khác từ khóa; TF-IDF vẫn hữu ích với tên riêng, mã số và truy vấn cần khớp chính xác.

## Đi sâu hơn: ma trận TF-IDF và phép tìm kiếm

Trong notebook buổi 2, context là danh sách N văn bản. Sau fit_transform, mỗi hàng là một văn bản và mỗi cột là một term trong vocabulary. Một query được transform thành một hàng có cùng số cột.

Ở notebook gốc, hàm tìm kiếm dựng lại vector context và đổi nó sang dense array ở mỗi query. Vì vectorizer đã được fit, có thể giữ document matrix một lần rồi tái sử dụng. Scikit-learn giữ ma trận TF-IDF ở dạng sparse để không phải cấp phát ô số 0 cho mọi cặp tài liệu/từ.

In [ ]:
# Bản tìm kiếm TF-IDF có thể tái sử dụng ma trận tài liệu
from sklearn.metrics.pairwise import cosine_similarity

doc_matrix = tfidf_vectorizer.fit_transform(context)
print("Số tài liệu, số chiều:", doc_matrix.shape)

def tfidf_search_reused(question, top_k=5):
    query_vector = tfidf_vectorizer.transform([question])
    scores = cosine_similarity(query_vector, doc_matrix).ravel()
    order = scores.argsort()[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in order]

example_query = context[0][:80]
tfidf_search_reused(example_query, top_k=3)

## Đọc output shape và điểm cosine

Nếu corpus có N đoạn và vocabulary có V từ, document matrix có shape (N, V). Query vector có shape (1, V). Cosine similarity trả N điểm — một điểm cho từng đoạn. Top-k là ID các điểm lớn nhất.

Nếu query không có từ nào trong vocabulary, vector query có thể toàn số 0 và mọi cosine score bằng 0. Khi đó việc argsort vẫn trả ID theo một thứ tự, nhưng không có đoạn nào thật sự khớp. Hãy kiểm tra score và cho phép “không đủ bằng chứng”.

## Dense embedding: kiểm tra hình học trước khi tải model

Embedding model nhận văn bản và trả vector dày. Trước khi dùng Sentence Transformers, luyện cách tính similarity với các vector nhỏ dưới đây. Số tọa độ chỉ là ví dụ; embedding thật không thể diễn giải bằng tay từng chiều.

In [ ]:
import numpy as np

query_demo = np.array([1.0, 0.0])
doc_demo = np.array([
    [1.0, 0.0],
    [0.8, 0.6],
    [0.0, 1.0]
])

norms = np.linalg.norm(doc_demo, axis=1)
scores = (doc_demo @ query_demo) / (norms * np.linalg.norm(query_demo))
list(zip(["A", "B", "C"], scores.round(3)))

## Bài tập phân tích lỗi

Tạo bốn query có chủ đích: từ khóa chính xác, paraphrase tiếng Việt, tên riêng/mã số và query không có đáp án. Với mỗi query, ghi ID đáp án trước khi chạy, top-3 TF-IDF, top-3 dense và lý do kết quả đúng/sai.

Không kết luận model tốt hơn chỉ vì một ví dụ. Với mỗi phương pháp, đọc cả đoạn nguồn và hỏi đoạn đó có trả lời câu hỏi không, hay chỉ cùng chủ đề.